# 01 — Opening a oneuniverse database

This notebook walks through the **first thing you do** with an oneuniverse install: pointing the package at a folder of converted survey catalogues, inspecting what is inside, and pulling a small chunk of data into memory.

**You will learn:**
1. How to instantiate `OneuniverseDatabase` from a root directory.
2. How the package discovers datasets and what each manifest contains.
3. How to read columns from a single dataset with `read_oneuniverse_parquet` (full read or column pushdown).
4. How to plot a sky map and a redshift histogram from one of the datasets.

---

**Prerequisite.** A folder containing one or more *converted* surveys, e.g.
```
<DB_ROOT>/spectroscopic/eboss_qso/oneuniverse/manifest.json
<DB_ROOT>/spectroscopic/eboss_qso/oneuniverse/part_0000.parquet
...
```
If you do not have one, you can convert a registered survey with `from oneuniverse.data import convert_survey`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from oneuniverse.data import OneuniverseDatabase
from oneuniverse.data.converter import read_oneuniverse_parquet

# ── Edit this path to point at your local database ──────────────────────
DB_ROOT = Path('/home/ravoux/Documents/Science/Cosmography/oneuniverse_database')

## 1. Instantiate the database

`OneuniverseDatabase` walks the folder tree (up to `max_depth=6` by default), reads every `oneuniverse/manifest.json` it finds, and registers a dynamically generated loader for each one. The dataset name is built from the relative path: `spectroscopic/eboss_qso/` becomes `spectroscopic_eboss_qso`.

In [ ]:
db = OneuniverseDatabase(DB_ROOT)
print(db.summary())

## 2. Listing and querying the registry

The database exposes a small dict-like API. You can iterate over dataset names, ask for the manifest of a single dataset, or get its on-disk path.

In [ ]:
# All datasets and their human-readable descriptions
for name, desc in db.list().items():
    print(f'  {name:40s}  {desc}')

# Group by survey type
print('\nSurvey types:', db.types())
print('Number of datasets:', len(db))

In [ ]:
# Pick the first dataset to explore in detail
dataset = next(iter(db))
print('Selected dataset:', dataset)

manifest = db.get_manifest(dataset)
for key in ('format_version', 'survey_name', 'survey_type', 'geometry',
            'n_rows', 'n_partitions', 'partition_rows', 'compression'):
    print(f'  {key:18s} = {manifest[key]}')

In [ ]:
# All columns available in this dataset
cols = manifest['data_columns']
print(f'{len(cols)} columns available:')
for c in cols:
    print(f'  {c}')

## 3. Reading data — full vs column pushdown

Two ways to materialise the Parquet partitions for a dataset:

* **Full read** — `read_oneuniverse_parquet(path)` loads every column from every partition. Convenient but expensive.
* **Pushdown read** — `read_oneuniverse_parquet(path, columns=[...])` only deserialises the requested columns. This is the recommended path for large catalogues; it scales linearly with the number of columns you actually need.

In [ ]:
import time

path = db.get_path(dataset)

# Pushdown read: only ra, dec, z (3 columns out of dozens)
t0 = time.perf_counter()
df_small = read_oneuniverse_parquet(path, columns=['ra', 'dec', 'z'])
t_small = time.perf_counter() - t0
print(f'pushdown 3 cols   : {t_small * 1e3:7.0f} ms  →  {df_small.shape}')
df_small.head()

In [ ]:
# Full read for comparison (skip if the dataset is huge)
if manifest['n_rows'] < 5_000_000:
    t0 = time.perf_counter()
    df_full = read_oneuniverse_parquet(path)
    t_full = time.perf_counter() - t0
    print(f'full read         : {t_full * 1e3:7.0f} ms  →  {df_full.shape}')
    print(f'speedup (push/full): {t_full / t_small:.1f}x')
else:
    print('Dataset is large — skipping the full read.')

## 4. Quick visual sanity check

Sky map (Mollweide) + redshift histogram. For 10⁶+ catalogues a `hist2d` is much faster than a scatter plot.

In [ ]:
ra = df_small['ra'].to_numpy()
dec = df_small['dec'].to_numpy()
z = df_small['z'].to_numpy()

fig = plt.figure(figsize=(13, 4.2))

# Sky density
ax = fig.add_subplot(1, 2, 1, projection='mollweide')
ra_rad = np.radians(np.where(ra > 180, ra - 360, ra))
dec_rad = np.radians(dec)
h = ax.hexbin(ra_rad, dec_rad, gridsize=80, cmap='inferno', bins='log',
              mincnt=1)
ax.set_title(f'{dataset}  —  {len(ra):,} objects on the sky')
ax.grid(alpha=0.3)
fig.colorbar(h, ax=ax, shrink=0.7, label='log₁₀ count')

# Redshift distribution
ax = fig.add_subplot(1, 2, 2)
z_finite = z[np.isfinite(z) & (z > 0)]
ax.hist(z_finite, bins=80, color='#3a7bd5', edgecolor='black', linewidth=0.4)
ax.set_xlabel('z')
ax.set_ylabel('count')
ax.set_title(f'Redshift distribution  ⟨z⟩={z_finite.mean():.2f}')

fig.tight_layout()
plt.show()

## 5. Going further

* **Selecting objects** with `Cone`, `Shell`, `SkyPatch` and the `OneuidQuery` API → see [`02_select_objects.ipynb`](02_select_objects.ipynb).
* **Cross-matching** several surveys to assign a universal `oneuid` → `db.build_oneuid()`.
* **Combining measurements** with the `oneuniverse.weight` strategies → `combine_weights(...)`.

If you want to register your own catalogue, write a `BaseSurveyLoader` subclass under `oneuniverse/data/surveys/<type>/<name>/loader.py`, then run `convert_survey(...)` to materialise it into the OUF Parquet layout.